# Qwen2.5-32B-Instruct Elevators (Python generator + review, bottom-to-top-delivery unraveling data)


In [ ]:
# Dependency install cell (one-shot setup for Kani + Qwen HF in the HazGPU virtual environment)
# Run once in a fresh kernel, then restart kernel.

import sys
import subprocess

py = sys.executable

def run(cmd):
  print(">", " ".join(cmd))
  subprocess.check_call(cmd)

# 1) Clean up known conflicting packages
run([py, "-m", "pip", "uninstall", "-y", "torchvision", "transformers", "tokenizers", "kani"])

# 2) Install a compatible set for kani HuggingEngine
run([py, "-m", "pip", "install", "--no-cache-dir", "-U",
   "kani[huggingface]==1.8.0",
   "transformers==4.51.3",
   "tokenizers==0.21.1",
   "accelerate>=0.30.0",
   "huggingface_hub>=0.30.0",
   "sentencepiece",
   "safetensors",
   "protobuf",
   "nest_asyncio",
   "python-dotenv",
   "tqdm"])

# 3) Ensure torch exists in the active virtual environment
try:
  import torch  # noqa: F401
  print("torch already available")
except Exception:
  run([py, "-m", "pip", "install", "--no-cache-dir", "-U", "torch"])

print("\nInstall complete. IMPORTANT: Restart kernel now, then run Cell 1.")

In [ ]:
# Quick test cell (run after kernel restart)

import os
import importlib.metadata as md

# Restrict this kernel to the assigned GPU on HazGPU before importing model libraries.
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["HF_HOME"] = "/local-ssd/hf_cache"

print("python:", __import__("sys").executable)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("HF_HOME:", os.environ["HF_HOME"])
for p in ["kani", "torch", "transformers", "accelerate", "tokenizers", "huggingface_hub"]:
  try:
      print(f"{p}: {md.version(p)}")
  except Exception:
      print(f"{p}: NOT INSTALLED")

from kani import Kani
from kani.engines.huggingface import HuggingEngine

print("Kani import OK")
print("HuggingEngine import OK")

In [ ]:
# Cell 1: Setup imports, paths, and Qwen2.5 Kani runtime (independent)
from pathlib import Path
import os
import re
import json
import time
import asyncio
import subprocess
import platform
from datetime import datetime

# These must be set before importing Kani or any library that may initialize CUDA.
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
os.environ['HF_HOME'] = '/local-ssd/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/local-ssd/hf_cache/hub'
os.environ['TORCH_HOME'] = '/local-ssd/oj58/.cache/torch'
os.environ['XDG_CACHE_HOME'] = '/local-ssd/oj58/.cache'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from dotenv import load_dotenv
from kani import Kani
from kani.engines.huggingface import HuggingEngine

# Project root (run notebook from formalizer-scaling/src/pddl_gen_notebooks/Elevators/bottom_to_top_delivery)
PROJECT_ROOT = Path.cwd().parents[3]

# Use the shared model cache and keep auxiliary caches on the assigned local SSD.
HF_HOME_DIR = Path(os.environ['HF_HOME'])
HF_HUB_CACHE_DIR = Path(os.environ['HUGGINGFACE_HUB_CACHE'])
USER_CACHE_ROOT = Path(os.environ['XDG_CACHE_HOME'])
TORCH_HOME_DIR = Path(os.environ['TORCH_HOME'])

for d in [USER_CACHE_ROOT, TORCH_HOME_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / 'data'
PROMPT_PATH = PROJECT_ROOT / 'prompts' / 'Elevators' / 'only_pddl_instruction_unraveling.txt'
PDDL_DIR = DATA_DIR / 'pddl_unraveling_domain' / 'Elevators' / 'bottom_to_top_delivery'
DESC_DIR = DATA_DIR / 'descriptions_unraveling_domain' / 'Elevators' / 'bottom_to_top_delivery'

OUTPUT_ROOT = PROJECT_ROOT / 'output' / 'direct_pddl_unraveling' / 'Elevators' / 'bottom_to_top_delivery' / 'qwen2_5_32b_instruct_kani_review_retry3_unraveling'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_BUILD_PLATFORM = 'macos64' if platform.system() == 'Darwin' else 'linux64'
VAL_PATH = PROJECT_ROOT / 'VAL' / 'build' / VAL_BUILD_PLATFORM / 'Release' / 'bin' / 'Validate'

# Model setup (Qwen2.5 Instruct via local HuggingFace + Kani)
MODEL_NAME = 'Qwen2.5-32B-Instruct'
HF_MODEL_ID = f'Qwen/{MODEL_NAME}'

QWEN_ENGINE = HuggingEngine(
    model_id=HF_MODEL_ID,
    model_load_kwargs={
        'device_map': 'auto',
        'cache_dir': str(HF_HUB_CACHE_DIR),
    },
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('VAL exists:', VAL_PATH.exists())
print('PDDL dir exists:', PDDL_DIR.exists())
print('Prompt exists:', PROMPT_PATH.exists())
print('Model:', HF_MODEL_ID)
print('CUDA_VISIBLE_DEVICES:', os.environ['CUDA_VISIBLE_DEVICES'])
print('HF cache root:', HF_HOME_DIR)

In [ ]:
# Cell 2: Define target problems and load static prompt text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 71)]  # p01..p70

PROMPT_PREFIX = PROMPT_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Description dir exists:", DESC_DIR.exists())
print("PDDL dir exists:", PDDL_DIR.exists())

In [ ]:
# Cell 3: Load NL descriptions and ground-truth PDDL for one problem ID
def load_problem_artifacts(problem_id: str):
  domain_nl_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_nl_path = DESC_DIR / f"{problem_id}_problem.txt"
  gt_domain_pddl_path = PDDL_DIR / "domain.pddl"
  gt_problem_pddl_path = PDDL_DIR / f"{problem_id}_problem.pddl"

  if not domain_nl_path.exists():
      raise FileNotFoundError(f"Missing domain NL file: {domain_nl_path}")
  if not problem_nl_path.exists():
      raise FileNotFoundError(f"Missing problem NL file: {problem_nl_path}")
  if not gt_domain_pddl_path.exists():
      raise FileNotFoundError(f"Missing ground-truth domain PDDL: {gt_domain_pddl_path}")
  if not gt_problem_pddl_path.exists():
      raise FileNotFoundError(f"Missing ground-truth problem PDDL: {gt_problem_pddl_path}")

  return {
      "nl_domain": domain_nl_path.read_text(),
      "nl_problem": problem_nl_path.read_text(),
      "gt_domain_pddl_path": gt_domain_pddl_path,
      "gt_problem_pddl_path": gt_problem_pddl_path,
      "gt_domain_pddl_text": gt_domain_pddl_path.read_text(),
  }

In [ ]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  artifacts = load_problem_artifacts(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer and expert Python engineer.\n"
      "Create an executable Python generator that writes a syntactically and semantically correct "
      "problem file using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{artifacts['nl_domain']}\n\n"
      "### Original problem description\n"
      f"{artifacts['nl_problem']}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{artifacts['gt_domain_pddl_text']}\n\n"
      "Output contract:\n"
      "- Generate ONLY an executable Python generator script.\n"
      "- Wrap it in exactly one <generator>...</generator> block.\n"
      "- Inside <generator>...</generator>, output ONLY raw Python source code.\n"
      "- The first non-whitespace character inside <generator> must be a valid Python token (e.g., def/import).\n"
      "- The script must write ONLY the final problem PDDL to 'problem.pddl' in current working directory.\n"
      "- The script MUST derive init/goal facts algorithmically from compact structure, not hardcoded full fact lists.\n"
      "- Required: use loops that construct repeated predicates and numeric assignments such as next, above, reachable-floor, can-hold, passenger-at, travel-slow, and travel-fast from computed patterns.\n"
      "- Required: compute count/floor, passenger, slow-elevator, and fast-elevator object lists from the inferred floor count and described fixed instance structure.\n"
      "- IMPORTANT: For F floors represented by n0 through n(F-1), generate every lower-floor/higher-floor combination for above, travel-slow, and travel-fast, every elevator/floor combination for reachable-floor, and every adjacent next relation. Do not omit combinations, reverse above arguments, or generate self-relations.\n"
      "- Disallowed: manually listing full init_state/goal_state facts as constants.\n"
      "- Do NOT include Markdown fences (no ```python, no ```).\n"
      "- Do NOT include comments or prose outside Python statements.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not output any <problem_file> block.\n"
      "- Do not include any extra text before or after <generator>...</generator>."
  )

In [ ]:
# Cell 5: Call Qwen2.5 via Kani for one prompt, then same-session generator review
GENERATOR_REVIEW_PROMPT_TEMPLATE = """Review the first draft of the Python generator against the original task and the fixed domain, then produce the corrected complete generator.

Previous generators have often produced plausible-looking instances that nevertheless contain missing required items, unsupported extra items, or information placed in the wrong category. Do not approve the draft merely because it runs. Before rewriting it, deliberately audit each of the following error categories.

Missing objects:
- Check that the object inventory contains every passenger, both slow elevators, the fast elevator, and every floor/count value required by the described building.
- Preserve every exact symbolic name and its exact category. The floor/count inventory must form one uninterrupted zero-based sequence from the bottom floor through the top floor.

Extra objects:
- Remove invented aliases, renamed elevators, generic replacements for the two elevator kinds, duplicate declarations, and objects not supported by the task.
- Make sure the program handles object names as collections of complete names. A prejoined string must never be iterated in a way that turns names into individual characters, and constructing a name must not repeat part of its prefix.

Missing initial-state information:
- Confirm that every passenger starts at the bottom, every elevator starts at the bottom, and every elevator initially carries zero passengers.
- Confirm that every elevator is associated with every floor in the building, with no elevator or floor omitted.
- Confirm that each slow elevator has every permitted positive passenger count from one through four and that the fast elevator has every permitted positive passenger count from one through six. The complete range is required, not only its maximum.
- Confirm that every neighboring pair across the full floor sequence is represented once in bottom-to-top order. This sequence covers the whole building and must not stop at an elevator capacity.
- Confirm that every distinct pair of floors is represented once with the lower floor first and the higher floor second. For each such pair, include both travel costs: the slow-elevator cost is the number of floors traveled plus five, and the fast-elevator cost is three times that distance plus one.
- Confirm that the initial objective value is present and begins at zero.

Extra or incorrect initial-state information:
- Remove reversed copies, self-pairs, duplicate relationships, duplicate travel entries, and any relationship whose order places the higher floor before the lower floor.
- Do not include zero as a permitted passenger count, do not add counts above an elevator's capacity, and do not attach information to invented or wrongly named elevators.
- Do not use ordinary English intuition to reverse the required lower-floor-first ordering. Do not create travel entries in both directions when only the lower-to-higher entry is required.
- Ensure that numerical values are written as actual assignments in the generated file rather than detached numbers, malformed fragments, or alternative notations.

Missing, extra, or shifted destination requirements:
- Include exactly one requested destination for every passenger described by the task, with no passenger missing, repeated, or added.
- Preserve each stated destination exactly. Convert a human-numbered floor to its zero-based internal floor exactly once; do not leave it one-based and do not subtract an additional step from a value already converted.

Completeness and generator reliability:
- Include the requested optimization objective. The generated file must be one complete instance, not a partial example, abbreviated pattern, or comment claiming that omitted cases should be filled in later.
- Inspect every loop bound and source collection for omissions and additions. Define every variable before use, keep repeated structures algorithmic, and ensure the program finishes successfully and writes the complete final problem file in the current working directory.
- Preserve parts of the draft that are already correct, but repair every missing, extra, malformed, reversed, mistyped, or shifted item by correcting the generator logic rather than manually expanding the instance.

Return only the complete corrected Python source in exactly one <generator>...</generator> block, with no text before or after it. Do not return the generated problem directly, and do not include Markdown fences or explanations.

Original task prompt:
{original_prompt}
"""


async def _call_qwen_async(prompt: str, temperature: float = 0.4, max_new_tokens: int = 32768):
  ai = Kani(QWEN_ENGINE, system_prompt='')
  await ai.chat_round_str(
      prompt,
      temperature=temperature,
      max_new_tokens=max_new_tokens,
  )
  review_prompt = GENERATOR_REVIEW_PROMPT_TEMPLATE.format(original_prompt=prompt)
  return await ai.chat_round_str(
      review_prompt,
      temperature=temperature,
      max_new_tokens=max_new_tokens,
  )


def call_qwen(prompt: str, temperature: float = 0.4, max_output_tokens: int = 32768):
  try:
      loop = asyncio.get_running_loop()
  except RuntimeError:
      return asyncio.run(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))

  import nest_asyncio
  nest_asyncio.apply()
  return loop.run_until_complete(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))


In [ ]:
# Cell 6: Parse model output and extract generator block
def extract_generator_code(raw_text: str):
  text = raw_text or ""
  matches = re.findall(r"<generator>(.*?)</generator>", text, flags=re.DOTALL | re.IGNORECASE)
  if not matches:
      return ""

  # Prefer the last block (models may emit multiple candidate blocks)
  code = (matches[-1] or "").strip()

  # Strip optional markdown fences if present
  if code.startswith("```"):
      code = re.sub(r"^```(?:python)?\s*", "", code, flags=re.IGNORECASE)
      code = re.sub(r"\s*```$", "", code)

  return code.strip()

In [ ]:
# Cell 7: Online solver (public endpoint) + raw API JSON persistence
import requests
import re
import json

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()
  res = None

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Persist full final API response for debugging/truncation analysis
  (work_dir / "solver_raw_response.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic for zero-cost empty-plan cases
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [ ]:
# Cell 7a: Revision gate (feedback-driven) for solver failures
def classify_solver_failure_for_revision(solver_result: dict) -> str:
  if solver_result.get("plan_found", False):
      return "plan_found"

  text = "\n".join(
      [
          str(solver_result.get("solver_error", "")),
          str(solver_result.get("stdout", "")),
          str(solver_result.get("stderr", "")),
      ]
  ).lower()

  search_started_markers = [
      "match tree built",
      "pddl problem description loaded",
      "starting search with",
      "--[",
      "goals found",
  ]
  search_finished_markers = [
      "plan found with cost",
      "search completed",
      "total time:",
      "nodes generated during search",
      "nodes expanded during search",
      "problem proven unsolvable",
      "no plan found",
  ]

  started = any(m in text for m in search_started_markers)
  finished = any(m in text for m in search_finished_markers)

  if started and not finished:
      return "cutoff_search_incomplete"   # do not revise
  if started and finished:
      return "completed_no_plan"          # do not revise

  return "pre_search_failure"             # revise


def should_run_revision(solver_result: dict) -> bool:
  return classify_solver_failure_for_revision(solver_result) == "pre_search_failure"


def build_revision_prompt(problem_id: str, previous_problem_pddl: str, previous_generator_code: str, solver_feedback: str) -> str:
  base = build_prompt(problem_id)
  return (
      base
      + "\n\nTo aid the revision, the previous generator code, previous generated problem PDDL, and solver feedback are provided below."
      + "\n------ Previous generator start ------\n"
      + "[GENERATOR CODE]\n"
      + (previous_generator_code or "")
      + "\n------ Previous generator end --------\n\n"
      + "------ Previous problem start ------\n"
      + "[PROBLEM FILE]\n"
      + (previous_problem_pddl or "")
      + "\n------ Previous problem end --------\n\n"
      + "Feedback from the planning solver:\n"
      + (solver_feedback or "Solver returned no plan.")
      + "\n\nRevise the Python generator to fix the issues."
      + " Output only the new generator wrapped in <generator>...</generator>."
  )

In [ ]:
# Cell 7b: Solver wrapper with feedback-driven revisions (up to 3) for generator workflow
import sys
import shutil

def _execute_generator_to_problem(generator_code: str, work_dir: Path, timeout_sec: int = 120):
  work_dir = Path(work_dir)
  work_dir.mkdir(parents=True, exist_ok=True)

  generator_path = work_dir / "generator.py"
  generator_path.write_text(generator_code or "")

  proc = subprocess.run(
      ["python", str(generator_path)],
      cwd=str(work_dir),
      capture_output=True,
      text=True,
      timeout=timeout_sec,
  )

  problem_path = work_dir / "problem.pddl"
  if not (problem_path.exists() and problem_path.read_text().strip()):
      # Fallback: if generator wrote a differently named problem file, copy latest candidate to problem.pddl
      candidates = [
          p for p in work_dir.rglob("*.pddl")
          if p.name not in {"domain.pddl", "problem.pddl"}
      ]
      if candidates:
          latest = max(candidates, key=lambda p: p.stat().st_mtime)
          shutil.copyfile(latest, problem_path)

  ok = problem_path.exists() and bool(problem_path.read_text().strip())
  return {
      "ok": ok and (proc.returncode == 0),
      "returncode": proc.returncode,
      "stdout": proc.stdout or "",
      "stderr": proc.stderr or "",
      "generator_path": str(generator_path),
      "problem_path": str(problem_path),
  }


def run_solver_with_revisions(
  problem_id: str,
  domain_pddl_path: Path,
  problem_pddl_path: Path,
  generator_path: Path,
  work_dir: Path,
  timeout_sec: int = 300,
  max_revisions: int = 3,
):
  problem_pddl_path = Path(problem_pddl_path)
  generator_path = Path(generator_path)
  work_dir = Path(work_dir)

  attempt = 0
  revisions_used = 0
  trace = []
  last_result = None

  while True:
      attempt += 1
      result = run_solver(
          domain_pddl_path=domain_pddl_path,
          problem_pddl_path=problem_pddl_path,
          work_dir=work_dir,
          timeout_sec=timeout_sec,
      )

      cls = classify_solver_failure_for_revision(result)
      print(
          f"[{problem_id}] revision_status: attempt={attempt}, class={cls}, plan_found={bool(result.get('plan_found', False))}",
          file=sys.stderr,
      )
      trace.append(
          {
              "attempt": attempt,
              "plan_found": bool(result.get("plan_found", False)),
              "classification": cls,
              "solver_error": str(result.get("solver_error", "")),
          }
      )
      last_result = result

      if result.get("plan_found", False):
          break

      # No revision for cutoff / completed-no-plan (by policy)
      if not should_run_revision(result):
          print(f"[{problem_id}] no revision triggered (class={cls})", file=sys.stderr)
          break

      if revisions_used >= max_revisions:
          print(f"[{problem_id}] max revisions reached ({max_revisions})", file=sys.stderr)
          break

      prev_problem_text = problem_pddl_path.read_text() if problem_pddl_path.exists() else ""
      prev_generator_text = generator_path.read_text() if generator_path.exists() else ""

      solver_feedback = (
          str(result.get("solver_error", "")).strip()
          or str(result.get("stderr", "")).strip()
          or str(result.get("stdout", "")).strip()
          or "Solver returned no plan."
      )

      print(f"[{problem_id}] revision {revisions_used + 1}/{max_revisions} started", file=sys.stderr)
      revision_prompt = build_revision_prompt(problem_id, prev_problem_text, prev_generator_text, solver_feedback)
      revision_raw = call_qwen(revision_prompt)
      revised_generator = extract_generator_code(revision_raw)

      revisions_used += 1
      rev_dir = work_dir / "revisions"
      rev_dir.mkdir(parents=True, exist_ok=True)
      (rev_dir / f"attempt_{revisions_used:02d}_prompt.txt").write_text(revision_prompt)
      (rev_dir / f"attempt_{revisions_used:02d}_raw.txt").write_text(revision_raw or "")

      if not (revised_generator or "").strip():
          fail = dict(result)
          fail["solver_error"] = (
              str(result.get("solver_error", "")).strip()
              + "\n[revision] model returned no <generator>...</generator> block."
          ).strip()
          last_result = fail
          trace[-1]["revision_generated"] = False
          print(
              f"[{problem_id}] revision {revisions_used}/{max_revisions} failed (no <generator>)",
              file=sys.stderr,
          )
          break

      exec_res = _execute_generator_to_problem(revised_generator, work_dir=work_dir, timeout_sec=120)

      (rev_dir / f"attempt_{revisions_used:02d}_generator.py").write_text(revised_generator)
      (rev_dir / f"attempt_{revisions_used:02d}_exec_stdout.txt").write_text(exec_res.get("stdout", ""))
      (rev_dir / f"attempt_{revisions_used:02d}_exec_stderr.txt").write_text(exec_res.get("stderr", ""))

      if not exec_res.get("ok", False):
          fail = dict(result)
          fail["solver_error"] = (
              str(result.get("solver_error", "")).strip()
              + "\n[revision] generator execution failed to produce valid problem.pddl."
          ).strip()
          last_result = fail
          trace[-1]["revision_generated"] = False
          trace[-1]["generator_exec_ok"] = False
          print(
              f"[{problem_id}] revision {revisions_used}/{max_revisions} failed (generator exec)",
              file=sys.stderr,
          )
          break

      # Persist latest generator and keep revised problem for next solver attempt
      generator_path.write_text(revised_generator)
      trace[-1]["revision_generated"] = True
      trace[-1]["generator_exec_ok"] = True
      print(
          f"[{problem_id}] revision {revisions_used}/{max_revisions} generated new generator.py/problem.pddl",
          file=sys.stderr,
      )

  if last_result is None:
      last_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "internal error: no solver attempts executed",
          "stdout": "",
          "stderr": "",
      }

  out = dict(last_result)
  out["attempts_used"] = attempt
  out["revisions_used"] = revisions_used
  out["retried"] = revisions_used > 0
  out["retry_trace"] = trace
  return out

In [ ]:
# Cell 8: Validate plan with VAL (terminal-equivalent capture, no timeout)
import subprocess
import shlex
import re

def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_file_path: Path):
  cmd = f"{shlex.quote(str(VAL_PATH))} {shlex.quote(str(domain_pddl_path))} {shlex.quote(str(problem_pddl_path))} {shlex.quote(str(plan_file_path))}"
  proc = subprocess.run(
      cmd,
      shell=True,
      capture_output=True,
      text=True,
  )

  val_output = (proc.stdout or "") + ("\n" + proc.stderr if proc.stderr else "")

  ok_patterns = [
      r"Plan valid",
      r"Successful plans:",
  ]
  bad_patterns = [
      r"Plan failed to execute",
      r"Bad plan description",
      r"Syntax error",
      r"Error",
      r"Failed",
      r"Unspecified precondition",
      r"goal not satisfied",
  ]

  val_ok = any(re.search(p, val_output, re.IGNORECASE) for p in ok_patterns) and not any(
      re.search(p, val_output, re.IGNORECASE) for p in bad_patterns
  )

  return {
      "val_ok": bool(val_ok),
      "val_output": val_output,
      "returncode": proc.returncode,
  }

In [ ]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, generator_code: str, problem_pddl_text: str, domain_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Generated Python generator code
  (cand_dir / "generator.py").write_text(generator_code or "")

  # Generated problem PDDL (after executing generator)
  (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(domain_pddl_text or "")

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  if "attempts_used" in solver_result:
      (cand_dir / "solver_attempts.txt").write_text(str(solver_result.get("attempts_used", 1)))
  if "retry_trace" in solver_result:
      (cand_dir / "solver_retry_trace.json").write_text(json.dumps(solver_result.get("retry_trace", []), indent=2))
  if "revisions_used" in solver_result:
      (cand_dir / "solver_revisions.txt").write_text(str(solver_result.get("revisions_used", 0)))

  return cand_dir

In [ ]:
# Cell 10: Run one problem through full pipeline
def process_one_problem(problem_id: str):
  artifacts = load_problem_artifacts(problem_id)
  prompt = build_prompt(problem_id)
  raw_text = call_qwen(prompt)

  generator_code = extract_generator_code(raw_text)

  # If model fails to return a tagged generator, record and stop early
  if not generator_code.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <generator>...</generator> found in model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_text, "", "", artifacts["gt_domain_pddl_text"], solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # Write generator first, then execute it to produce problem.pddl
  problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
  problem_dir.mkdir(parents=True, exist_ok=True)

  exec_result = _execute_generator_to_problem(generator_code, work_dir=problem_dir, timeout_sec=120)

  (problem_dir / "generator_exec_stdout.txt").write_text(exec_result.get("stdout", "") or "")
  (problem_dir / "generator_exec_stderr.txt").write_text(exec_result.get("stderr", "") or "")

  if not exec_result.get("ok", False):
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "Generator execution failed to produce valid problem.pddl.",
          "stdout": exec_result.get("stdout", "") or "",
          "stderr": exec_result.get("stderr", "") or "",
      }
      final_problem_pddl_text = (problem_dir / "problem.pddl").read_text() if (problem_dir / "problem.pddl").exists() else ""
      save_problem_outputs(problem_id, raw_text, generator_code, final_problem_pddl_text, artifacts["gt_domain_pddl_text"], solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  gen_problem_path = problem_dir / "problem.pddl"
  gt_domain_pddl_path = artifacts["gt_domain_pddl_path"]
  gt_problem_pddl_path = artifacts["gt_problem_pddl_path"]

  solver_result = run_solver_with_revisions(
      problem_id=problem_id,
      domain_pddl_path=gt_domain_pddl_path,
      problem_pddl_path=gen_problem_path,
      generator_path=problem_dir / "generator.py",
      work_dir=problem_dir,
      timeout_sec=300,
      max_revisions=3,
  )

  val_result = None
  val_ok = False
  if solver_result["plan_found"]:
      plan_path = problem_dir / "plan"
      if not plan_path.exists():
          # fallback safety if solver wrote plan text but not file
          plan_path = problem_dir / "plan.txt"
          plan_path.write_text(solver_result["plan_text"])
      val_result = run_val(gt_domain_pddl_path, gt_problem_pddl_path, plan_path)
      val_ok = bool(val_result["val_ok"])

  # Persist final artifacts (keep latest revised generator/problem if revisions occurred)
  final_problem_pddl_text = gen_problem_path.read_text() if gen_problem_path.exists() else ""
  final_generator_code = (problem_dir / "generator.py").read_text() if (problem_dir / "generator.py").exists() else generator_code
  save_problem_outputs(problem_id, raw_text, final_generator_code, final_problem_pddl_text, artifacts["gt_domain_pddl_text"], solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result["plan_found"]),
      "val_ok": val_ok,
      "solver_error": solver_result.get("solver_error", ""),
      "attempts_used": int(solver_result.get("attempts_used", 1)),
  }

In [ ]:
# Cell 11: Generate only with review (no solver/VAL)
smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)

prompt = build_prompt(smoke_problem)
raw_text = call_qwen(prompt)
generator_code = extract_generator_code(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

exec_result = {"ok": False, "stdout": "", "stderr": ""}
problem_pddl_text = ""

if generator_code.strip():
  exec_result = _execute_generator_to_problem(generator_code, work_dir=cand_dir, timeout_sec=120)
  if (cand_dir / "problem.pddl").exists():
      problem_pddl_text = (cand_dir / "problem.pddl").read_text()

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "generator.py").write_text(generator_code or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
(cand_dir / "generator_exec_stdout.txt").write_text(exec_result.get("stdout", "") or "")
(cand_dir / "generator_exec_stderr.txt").write_text(exec_result.get("stderr", "") or "")

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_generator_chars:", len(generator_code or ""))
print("generated_problem_pddl_chars:", len(problem_pddl_text or ""))
print("generator_exec_ok:", bool(exec_result.get("ok", False)))
print("saved:", cand_dir)

In [ ]:
# Cell 12: Solver+VAL (+optional revision regeneration), concise output -- VAL uses ground-truth domain/problem files
import contextlib
import io

smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

problem_path = cand_dir / "problem.pddl"   # generated problem for solver
generator_path = cand_dir / "generator.py" # generated python generator
if not problem_path.exists() or not problem_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty generated problem file: {problem_path}")
if not generator_path.exists() or not generator_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty generator file: {generator_path}")

gt_domain_path = artifacts["gt_domain_pddl_path"]
gt_problem_path = artifacts["gt_problem_pddl_path"]

# Suppress verbose live logs from solver in notebook output
with contextlib.redirect_stdout(io.StringIO()):
  solver_result = run_solver_with_revisions(
      problem_id=smoke_problem,
      domain_pddl_path=gt_domain_path,
      problem_pddl_path=problem_path,
      generator_path=generator_path,
      work_dir=cand_dir,
      timeout_sec=2000,
      max_revisions=3,
  )

# If revisions happened, refresh final generated files from disk
final_problem_pddl_text = problem_path.read_text() if problem_path.exists() else ""
final_generator_code = generator_path.read_text() if generator_path.exists() else ""

# Save solver logs/results
(cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
(cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

produced_files = ["solver_stdout.txt", "solver_stderr.txt"]
val_result = None

if solver_result["plan_found"]:
  plan_file = cand_dir / "plan.txt"
  plan_text = solver_result.get("plan_text", "")
  plan_file.write_text(plan_text)
  produced_files.append("plan.txt")

  with contextlib.redirect_stdout(io.StringIO()):
      val_result = run_val(gt_domain_path, gt_problem_path, plan_file)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
  produced_files.append("validator.log")
else:
  (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
  produced_files.append("solver_error.txt")

save_problem_outputs(
  problem_id=smoke_problem,
  raw_text=(cand_dir / "raw.txt").read_text() if (cand_dir / "raw.txt").exists() else "",
  generator_code=final_generator_code,
  problem_pddl_text=final_problem_pddl_text,
  domain_pddl_text=artifacts["gt_domain_pddl_text"],
  solver_result=solver_result,
  val_result=val_result,
)

summary = {
  "problem": smoke_problem,
  "plan_found": bool(solver_result["plan_found"]),
  "val_ok": bool(val_result["val_ok"]) if val_result is not None else False,
  "solver_error": solver_result.get("solver_error", ""),
  "attempts_used": int(solver_result.get("attempts_used", 1)),
  "revisions_used": int(solver_result.get("revisions_used", 0)),
}

print("Produced files:")
for name in produced_files:
  print("-", name)

if val_result is not None:
  print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
else:
  print("VAL result: not run (solver failed)")

print("Summary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 13: Batch generation-only with review (with tqdm), default p02..p70
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(2, 71)]  # p02..p70; p01 is generated by the smoke cell

gen_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="LLM generation", unit="problem"):
  try:
      artifacts = load_problem_artifacts(pid)
      prompt = build_prompt(pid)
      raw_text = call_qwen(prompt)
      generator_code = extract_generator_code(raw_text)

      cand_dir = OUTPUT_ROOT / pid / "cand_00"
      cand_dir.mkdir(parents=True, exist_ok=True)

      exec_result = {"ok": False, "stdout": "", "stderr": ""}
      problem_pddl_text = ""

      if generator_code.strip():
          exec_result = _execute_generator_to_problem(generator_code, work_dir=cand_dir, timeout_sec=120)
          if (cand_dir / "problem.pddl").exists():
              problem_pddl_text = (cand_dir / "problem.pddl").read_text()

      (cand_dir / "raw.txt").write_text(raw_text or "")
      (cand_dir / "generator.py").write_text(generator_code or "")
      (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
      (cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
      (cand_dir / "generator_exec_stdout.txt").write_text(exec_result.get("stdout", "") or "")
      (cand_dir / "generator_exec_stderr.txt").write_text(exec_result.get("stderr", "") or "")

      ok = bool(generator_code.strip()) and bool(exec_result.get("ok", False)) and bool(problem_pddl_text.strip())
      gen_results.append({"problem": pid, "generation_ok": ok, "error": ""})

      if ok:
          print(f"{pid}: generation ok")
      else:
          print(f"{pid}: generation failed (generator missing or execution failed)")
  except Exception as e:
      gen_results.append({"problem": pid, "generation_ok": False, "error": str(e)})
      print(f"{pid}: generation error -> {e}")

print("\nGeneration batch done.")
print("total:", len(gen_results))
print("ok:", sum(1 for r in gen_results if r["generation_ok"]))
print("failed:", sum(1 for r in gen_results if not r["generation_ok"]))

In [ ]:
# Cell 14: Batch solver+VAL only (self-contained, with its own problem handle)
from tqdm.auto import tqdm
import contextlib
import io

BATCH_PROBLEMS_SOLVER_VAL = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 11)]  # change as needed

sv_results = []
for pid in tqdm(BATCH_PROBLEMS_SOLVER_VAL, desc="Solver+VAL", unit="problem"):
  artifacts = load_problem_artifacts(pid)
  cand_dir = OUTPUT_ROOT / pid / "cand_00"
  problem_path = cand_dir / "problem.pddl"
  generator_path = cand_dir / "generator.py"

  if not problem_path.exists() or not problem_path.read_text().strip():
      msg = f"missing/empty problem file: {problem_path}"
      sv_results.append({"problem": pid, "plan_found": False, "val_ok": False, "error": msg})
      print(f"{pid}: skipped ({msg})")
      continue

  if not generator_path.exists() or not generator_path.read_text().strip():
      msg = f"missing/empty generator file: {generator_path}"
      sv_results.append({"problem": pid, "plan_found": False, "val_ok": False, "error": msg})
      print(f"{pid}: skipped ({msg})")
      continue

  gt_domain_path = artifacts["gt_domain_pddl_path"]
  gt_problem_path = artifacts["gt_problem_pddl_path"]

  # Suppress verbose live logs in batch mode
  with contextlib.redirect_stdout(io.StringIO()):
      solver_result = run_solver_with_revisions(
          problem_id=pid,
          domain_pddl_path=gt_domain_path,
          problem_pddl_path=problem_path,
          generator_path=generator_path,
          work_dir=cand_dir,
          timeout_sec=300,
          max_revisions=3,
      )

  # Refresh files in case revisions occurred
  final_problem_pddl_text = problem_path.read_text() if problem_path.exists() else ""
  final_generator_code = generator_path.read_text() if generator_path.exists() else ""

  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  val_ok = False
  val_result = None
  if solver_result.get("plan_found"):
      plan_file = cand_dir / "plan.txt"
      plan_file.write_text(solver_result.get("plan_text", "") or "")

      with contextlib.redirect_stdout(io.StringIO()):
          val_result = run_val(gt_domain_path, gt_problem_path, plan_file)

      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
      val_ok = bool(val_result.get("val_ok", False))
      print(f"{pid}: val completed ({'valid' if val_ok else 'invalid'})")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
      print(f"{pid}: solver failed")

  save_problem_outputs(
      problem_id=pid,
      raw_text=(cand_dir / "raw.txt").read_text() if (cand_dir / "raw.txt").exists() else "",
      generator_code=final_generator_code,
      problem_pddl_text=final_problem_pddl_text,
      domain_pddl_text=artifacts["gt_domain_pddl_text"],
      solver_result=solver_result,
      val_result=val_result,
  )

  sv_results.append(
      {
          "problem": pid,
          "plan_found": bool(solver_result.get("plan_found", False)),
          "val_ok": val_ok,
          "error": solver_result.get("solver_error", ""),
          "attempts_used": int(solver_result.get("attempts_used", 1)),
          "revisions_used": int(solver_result.get("revisions_used", 0)),
      }
  )

print("\nSolver+VAL batch done.")
print("total:", len(sv_results))
print("plan_found:", sum(1 for r in sv_results if r["plan_found"]))
print("val_ok:", sum(1 for r in sv_results if r["val_ok"]))
print("failed:", sum(1 for r in sv_results if not r["val_ok"]))


In [ ]:
# Cell 15: In-notebook metrics summary (no persistence)
import pandas as pd

# Define any batch you want here
SUMMARY_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 11)]  # example: p01..p10

rows = []
for pid in SUMMARY_PROBLEMS:
  cand_dir = OUTPUT_ROOT / pid / "cand_00"

  solver_stdout_path = cand_dir / "solver_stdout.txt"
  solver_stderr_path = cand_dir / "solver_stderr.txt"
  solver_error_path = cand_dir / "solver_error.txt"
  plan_txt_path = cand_dir / "plan.txt"
  plan_path = cand_dir / "plan"
  val_log_path = cand_dir / "validator.log"
  generator_path = cand_dir / "generator.py"
  problem_pddl_path = cand_dir / "problem.pddl"

  solver_stdout = solver_stdout_path.read_text() if solver_stdout_path.exists() else ""
  solver_stderr = solver_stderr_path.read_text() if solver_stderr_path.exists() else ""
  solver_error = solver_error_path.read_text().strip() if solver_error_path.exists() else ""

  has_plan_txt = plan_txt_path.exists() and plan_txt_path.read_text().strip() != ""
  has_plan = plan_path.exists() and plan_path.read_text().strip() != ""
  plan_found = bool(has_plan_txt or has_plan)

  val_log = val_log_path.read_text() if val_log_path.exists() else ""
  val_ran = val_log_path.exists() and val_log.strip() != ""
  val_ok = "Plan valid" in val_log

  if val_ok:
      status = "plan_valid"
  elif plan_found and val_ran:
      status = "plan_invalid"
  else:
      status = "solver_failed"

  rows.append(
      {
          "problem_id": pid,
          "status": status,
          "plan_found": plan_found,
          "val_ran": val_ran,
          "val_ok": val_ok,
          "generator_exists": generator_path.exists(),
          "problem_pddl_exists": problem_pddl_path.exists() and problem_pddl_path.read_text().strip() != "",
          "solver_error": solver_error,
          "solver_stdout_chars": len(solver_stdout),
          "solver_stderr_chars": len(solver_stderr),
          "cand_dir": str(cand_dir),
      }
  )

details_df = pd.DataFrame(rows).sort_values("problem_id").reset_index(drop=True)

total = len(details_df)
plan_found_n = int(details_df["plan_found"].sum())
val_ok_n = int(details_df["val_ok"].sum())
solver_failed_n = int((details_df["status"] == "solver_failed").sum())
plan_invalid_n = int((details_df["status"] == "plan_invalid").sum())

summary_df = pd.DataFrame(
  [
      {
          "num_problems": total,
          "plan_found_count": plan_found_n,
          "plan_found_rate": plan_found_n / total if total else 0.0,
          "val_ok_count": val_ok_n,
          "val_ok_rate": val_ok_n / total if total else 0.0,
          "solver_failed_count": solver_failed_n,
          "plan_invalid_count": plan_invalid_n,
      }
  ]
)

print("Summary:")
display(summary_df)

print("Per-problem details:")
display(details_df[["problem_id", "status", "plan_found", "val_ran", "val_ok", "solver_error"]])

# Keeps objects in memory for Cell 16 persistence
SUMMARY_DF = summary_df
DETAILS_DF = details_df
SUMMARY_PROBLEMS_USED = SUMMARY_PROBLEMS

In [ ]:
# Cell 16: Persist summary/details to CSV (run only when you want to save)
from datetime import datetime

# Optional label to distinguish runs (edit as needed)
SAVE_LABEL = "qwen2_5_32b_instruct_kani_review_python_unraveling_elevators"

# Where to save metric CSVs
METRIC_DIR = OUTPUT_ROOT / "_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if "SUMMARY_DF" not in globals() or "DETAILS_DF" not in globals():
  raise RuntimeError("Run Cell 15 first to populate SUMMARY_DF and DETAILS_DF.")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
problems_tag = f"{SUMMARY_PROBLEMS_USED[0]}_to_{SUMMARY_PROBLEMS_USED[-1]}" if len(SUMMARY_PROBLEMS_USED) > 1 else SUMMARY_PROBLEMS_USED[0]

summary_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_summary_{ts}.csv"
details_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_details_{ts}.csv"

SUMMARY_DF.to_csv(summary_out, index=False)
DETAILS_DF.to_csv(details_out, index=False)

print("Saved:")
print("-", summary_out)
print("-", details_out)

# Parser for Problem File & Difference Checking


In [ ]:
# Cell 17: Diff section setup (independent, PDDL-vs-PDDL for unraveling)
from pathlib import Path
import re
import json

PROJECT_ROOT = Path.cwd().parents[3]  # run notebook from formalizer-scaling/src/pddl_gen_notebooks/Elevators/bottom_to_top_delivery
PDDL_DIR = PROJECT_ROOT / "data" / "pddl_unraveling_domain" / "Elevators" / "bottom_to_top_delivery"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_unraveling" / "Elevators" / "bottom_to_top_delivery" / "qwen2_5_32b_instruct_kani_review_retry3_unraveling"

# Single-problem handle
DIFF_PROBLEM = "p01"  # edit as needed

# Input files
GT_PDDL_PATH = PDDL_DIR / f"{DIFF_PROBLEM}_problem.pddl"
LLM_PDDL_PATH = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00" / "problem.pddl"

# Output files
CAND_DIR = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00"
CAND_DIR.mkdir(parents=True, exist_ok=True)

DIFF_JSON_OUT = CAND_DIR / "gt_vs_llm_diff.json"
DIFF_TXT_OUT = CAND_DIR / "gt_vs_llm_diff.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem:", DIFF_PROBLEM)
print("GT PDDL exists:", GT_PDDL_PATH.exists())
print("LLM PDDL exists:", LLM_PDDL_PATH.exists())
print("Output dir:", CAND_DIR)

In [ ]:
# Cell 18: Helper 1 - normalize whitespace, canonicalize Elevators symbols, and extract top-level sections
import re


def _norm_space(s: str) -> str:
  return re.sub(r"\s+", " ", s.strip())


def _canon_symbol(sym: str) -> str:
  s = sym.strip().lower()

  m = re.fullmatch(r"(?:n|count)-?(\d+)", s)
  if m:
      return f"n{int(m.group(1))}"

  m = re.fullmatch(r"floor-?(\d+)", s)
  if m:
      floor_number = int(m.group(1))
      return f"n{floor_number - 1}" if floor_number >= 1 else s

  m = re.fullmatch(r"(?:p|passenger|person)-?(\d+)", s)
  if m:
      return f"p{int(m.group(1))}"

  m = re.fullmatch(r"(?:slow|slow-elevator)-?(\d+)", s)
  if m:
      return f"slow{int(m.group(1))}"

  m = re.fullmatch(r"(?:fast|fast-elevator)-?(\d+)", s)
  if m:
      return f"fast{int(m.group(1))}"

  return s


def _extract_section_body(pddl_text: str, section_name: str) -> str:
  marker = f"(:{section_name}"
  start = pddl_text.lower().find(marker)
  if start == -1:
      raise ValueError(f"PDDL missing :{section_name} section.")

  i = start + len(marker)
  depth = 1
  body_chars = []

  while i < len(pddl_text) and depth > 0:
      ch = pddl_text[i]
      if ch == "(":
          depth += 1
      elif ch == ")":
          depth -= 1
          if depth == 0:
              break
      body_chars.append(ch)
      i += 1

  if depth != 0:
      raise ValueError(f"Unbalanced parentheses while reading :{section_name} section.")

  return "".join(body_chars)


In [ ]:
# Cell 19: Helper 2 - extract Elevators objects/init/goal from a PDDL problem file
_PREDICATE_RE = re.compile(
    r"\((passenger-at|boarded|lift-at|reachable-floor|above|passengers|can-hold|next)\s+([^()]+?)\)",
    flags=re.IGNORECASE,
)
_TRAVEL_COST_RE = re.compile(
    r"\(=\s*\((travel-slow|travel-fast)\s+([^()\s]+)\s+([^()\s]+)\)\s+([^()\s]+)\)",
    flags=re.IGNORECASE,
)
_TOTAL_COST_RE = re.compile(
    r"\(=\s*\(total-cost\)\s+([^()\s]+)\)",
    flags=re.IGNORECASE,
)


def _extract_typed_objects(objects_text: str):
  tokens = _norm_space(objects_text).split()
  object_decls = []
  pending_names = []
  i = 0

  while i < len(tokens):
      token = tokens[i]
      if token == "-":
          if not pending_names or i + 1 >= len(tokens):
              raise ValueError("Malformed typed object declaration.")
          obj_type = tokens[i + 1].lower()
          object_decls.extend((_canon_symbol(name), obj_type) for name in pending_names)
          pending_names = []
          i += 2
      else:
          pending_names.append(token)
          i += 1

  if pending_names:
      object_decls.extend((_canon_symbol(name), "object") for name in pending_names)

  return object_decls


def _extract_predicate_facts(section_text: str):
  facts = []
  for match in _PREDICATE_RE.finditer(section_text):
      predicate = match.group(1).lower()
      arguments = [_canon_symbol(arg) for arg in _norm_space(match.group(2)).split()]
      facts.append(tuple([predicate] + arguments))
  return facts


def extract_elevators_problem_structure(pddl_text: str):
  objects_text = _extract_section_body(pddl_text, "objects")
  init_text = _extract_section_body(pddl_text, "init")
  goal_text = _extract_section_body(pddl_text, "goal")

  init_facts = _extract_predicate_facts(init_text)
  for match in _TRAVEL_COST_RE.finditer(init_text):
      init_facts.append((
          match.group(1).lower(),
          _canon_symbol(match.group(2)),
          _canon_symbol(match.group(3)),
          match.group(4).strip(),
      ))
  for match in _TOTAL_COST_RE.finditer(init_text):
      init_facts.append(("total-cost", match.group(1).strip()))

  return {
      "objects": _extract_typed_objects(objects_text),
      "init_facts": init_facts,
      "goal_facts": _extract_predicate_facts(goal_text),
  }


In [ ]:
# Cell 20: Helper 3 - Elevators, order-insensitive structural diff (GT vs LLM)
def diff_fact_sections(gt_pddl_text: str, llm_pddl_text: str):
  gt = extract_elevators_problem_structure(gt_pddl_text)
  llm = extract_elevators_problem_structure(llm_pddl_text)

  gt_objects = set(gt["objects"])
  llm_objects = set(llm["objects"])
  gt_init = set(gt["init_facts"])
  llm_init = set(llm["init_facts"])
  gt_goal = set(gt["goal_facts"])
  llm_goal = set(llm["goal_facts"])

  return {
      "objects_missing_in_llm": sorted(gt_objects - llm_objects),
      "objects_extra_in_llm": sorted(llm_objects - gt_objects),
      "init_missing_in_llm": sorted(gt_init - llm_init),
      "init_extra_in_llm": sorted(llm_init - gt_init),
      "goal_missing_in_llm": sorted(gt_goal - llm_goal),
      "goal_extra_in_llm": sorted(llm_goal - gt_goal),
      "objects_match_count": len(gt_objects & llm_objects),
      "init_match_count": len(gt_init & llm_init),
      "goal_match_count": len(gt_goal & llm_goal),
      "objects_total_gt": len(gt_objects),
      "objects_total_llm": len(llm_objects),
      "init_total_gt": len(gt_init),
      "init_total_llm": len(llm_init),
      "goal_total_gt": len(gt_goal),
      "goal_total_llm": len(llm_goal),
      "perfect_match": (
          gt_objects == llm_objects and
          gt_init == llm_init and
          gt_goal == llm_goal
      ),
  }

In [ ]:
# Cell 21: Helper 4 - concise mismatch-only diff report text
def build_diff_report_text(problem_id: str, diff_result: dict):
  lines = []
  lines.append(f"Problem: {problem_id}")
  lines.append(f"Perfect match: {diff_result['perfect_match']}")
  lines.append("")

  lines.append(f"Objects missing in LLM ({len(diff_result['objects_missing_in_llm'])}):")
  for obj in diff_result["objects_missing_in_llm"]:
      lines.append(f"- {obj}")

  lines.append("")
  lines.append(f"Objects extra in LLM ({len(diff_result['objects_extra_in_llm'])}):")
  for obj in diff_result["objects_extra_in_llm"]:
      lines.append(f"- {obj}")

  lines.append("")
  lines.append(f"Init missing in LLM ({len(diff_result['init_missing_in_llm'])}):")
  for fact in diff_result["init_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Init extra in LLM ({len(diff_result['init_extra_in_llm'])}):")
  for fact in diff_result["init_extra_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal missing in LLM ({len(diff_result['goal_missing_in_llm'])}):")
  for fact in diff_result["goal_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal extra in LLM ({len(diff_result['goal_extra_in_llm'])}):")
  for fact in diff_result["goal_extra_in_llm"]:
      lines.append(f"- {fact}")

  return "\n".join(lines) + "\n"

In [ ]:
# Cell 22: Diff run (single problem handle, persists outputs to pXXX/cand_00)
if not GT_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing ground-truth PDDL: {GT_PDDL_PATH}")
if not LLM_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing LLM-generated PDDL: {LLM_PDDL_PATH}")

gt_pddl_text = GT_PDDL_PATH.read_text()
llm_pddl_text = LLM_PDDL_PATH.read_text()

diff_result = diff_fact_sections(gt_pddl_text, llm_pddl_text)
diff_text = build_diff_report_text(DIFF_PROBLEM, diff_result)

DIFF_JSON_OUT.write_text(json.dumps(diff_result, indent=2))
DIFF_TXT_OUT.write_text(diff_text)

print("Saved:")
print("-", DIFF_JSON_OUT)
print("-", DIFF_TXT_OUT)
print("")
print("Mismatch summary:")
print(f"- objects missing: {len(diff_result['objects_missing_in_llm'])}")
print(f"- objects extra:   {len(diff_result['objects_extra_in_llm'])}")
print(f"- init missing:    {len(diff_result['init_missing_in_llm'])}")
print(f"- init extra:      {len(diff_result['init_extra_in_llm'])}")
print(f"- goal missing:    {len(diff_result['goal_missing_in_llm'])}")
print(f"- goal extra:      {len(diff_result['goal_extra_in_llm'])}")
print(f"- perfect_match: {diff_result['perfect_match']}")